# 01 · Download ERA5-Land monthly data

We download monthly means of **2 m temperature** (`t2m`, K) and **total precipitation** (`tp`, m/day)
from the Copernicus Climate Data Store (CDS) with `cdsapi`, for a box around the catchment and 1981–2020.

Before running: create a CDS account, accept the ERA5-Land licence and save your API key in `~/.cdsapirc`.
The download can wait in the CDS queue for a while; if the file already exists it is not downloaded again.

The monthly means of daily maximum and minimum temperature are provided in `data/raw/tmax_tmin.nc`.

In [1]:
%load_ext autoreload
%autoreload 2                  # reload edited src/epss modules automatically
import sys, json
from pathlib import Path
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("../data")
PROC = DATA / "processed"
PROC.mkdir(exist_ok=True)
case = json.loads((DATA / "case.json").read_text())
case

{'catchment_area_km2': 55.33,
 'latitude_deg': 38.237,
 'longitude_deg': -7.687,
 'smax_mm': 100,
 'hmax_m': 5.0,
 'hmin_m': 1.0,
 'area_fixed_km2': 0.0685,
 'irrigation_months': [6, 7, 8, 9],
 'demand_hm3_per_month': 0.02,
 'eflow_fraction': 0.1,
 'target_reliability': 0.9}

The request: dataset `reanalysis-era5-land-monthly-means`, product `monthly_averaged_reanalysis`,
area `[N, W, S, E]` = catchment centre ± 1.5° (reaches the coast).

In [2]:
target = DATA / "raw" / "era5land_monthly.nc"
lat, lon = case["latitude_deg"], case["longitude_deg"]
area = [round(lat + 1.5, 1), round(lon - 1.5, 1), round(lat - 1.5, 1), round(lon + 1.5, 1)]

if not target.exists():
    import cdsapi
    client = cdsapi.Client()          # reads ~/.cdsapirc
    # TODO: Request ERA5-Land monthly means (dataset 'reanalysis-era5-land-monthly-means', product
    #       'monthly_averaged_reanalysis') of 2m_temperature and total_precipitation, 1981-2020, all
    #       months, time 00:00, for `area`, as NetCDF.
    # In: area [N, W, S, E], years, variables
    # Out: NetCDF file saved at `target`
    raise NotImplementedError("Request ERA5-Land monthly means (dataset 'reanalysis-era5-land-monthly-means', product 'monthly_averaged_reanalysis') of 2m_temperature and total_precipitation, 1981-2020, all months, time 00:00, for `area`, as NetCDF.")
print(target, target.exists())

..\data\raw\era5land_monthly.nc True


In [3]:
import xarray as xr
ds = xr.open_dataset(target)
ds

<xarray.Dataset> Size: 3MB
Dimensions:     (valid_time: 480, latitude: 29, longitude: 31)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 4kB 1981-01-01 ... 2020-12-01
  * latitude    (latitude) float64 232B 39.7 39.6 39.5 39.4 ... 37.1 37.0 36.9
  * longitude   (longitude) float64 248B -9.2 -9.1 -9.0 -8.9 ... -6.4 -6.3 -6.2
Data variables:
    t2m         (valid_time, latitude, longitude) float32 2MB ...
    tp          (valid_time, latitude, longitude) float32 2MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-03T22:21 GRIB to CDM+CF via cfgrib-0.9.1...

Mean maps of the region with `meteoraster`: coastline, borders and the catchment (`catchment.kml`).
`to_meteoraster` wraps the NetCDF grid in a `MeteoRaster`; precipitation is converted to mm/year
(mean monthly total × 12).

In [ ]:
import cartopy.crs as ccrs
from epss.climate import to_meteoraster, catchment_polygon

catchment = catchment_polygon(DATA / "catchment.kml")
maps = []
for var, cmap in [("t2m", "RdYlBu_r"), ("tp", "Blues")]:
    da = ds[var]
    units = da.attrs.get("units", "")
    label = f"Mean {da.attrs.get('long_name', var)} ({units})"
    maps.append((to_meteoraster(da, var, units), label, cmap))
for mr, label, cmap in maps:
    ax = mr.create_plot(central_longitude=0, figsize=(6, 5))
    ax.add_geometries(catchment, crs=ccrs.PlateCarree(), facecolor="none", edgecolor="k", linewidth=1.5)
    ax.gridlines(draw_labels=["left", "bottom"], linestyle=":", color="grey")
    ax.set_title(label)                       # set before plot_mean, which calls plt.show()
    mr.plot_mean(ax=ax, coastline=True, borders=True, colorbar_label=label, cmap=cmap, add_labels=False)